import os, warnings, torch

import scanpy as sc
import pandas as pd

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


In [ ]:
import os, warnings, torch

import scanpy as sc
import pandas as pd

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


### Load dataset

In [ ]:
path = '/home/wzk/ST_code/NicheTrans/2024_NicheTrans_upload_data/2023_nn_AD_mouse/wild_type_adata_protein/spatial_13months-control-replicate_1.h5ad'
rna_adata = sc.read_h5ad(path)

### Load args

In [ ]:
%run ./args/args_STARmap_PLUS.py
args = args

### Create dataloader

In [ ]:
# create the dataset
dataset = AD_Mouse(
    AD_adata_path=args.AD_adata_path,
    Wild_type_adata_path=args.Wild_type_adata_path,
    n_top_genes=args.n_top_genes,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Model initialization

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=False).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_STARmap_PLUS.pth', device=device)


### Model inference 

In [ ]:
pd_value, gt_value, _ = collect_predictions(
    model,
    dataset.val,
    split='val',
    device=device,
    apply_sigmoid=True,
)

pd_value = pd_value.numpy()
gt_value = gt_value.numpy()


In [ ]:
rna_adata.obs['pd_tau'] = (pd_value[:, 0] > 0.5) * 1
rna_adata.obs['pd_plaque'] = (pd_value[:, 1] > 0.5) * 1

rna_adata.obs['tau'] = (gt_value[:, 0] > 0.5) * 1
rna_adata.obs['plaque'] = (gt_value[:, 1] > 0.5) * 1

### Model evaluation

In [ ]:
key_pd0, key_gt0 = 'pd_' + dataset.target_panel[0], dataset.target_panel[0]
key_pd1, key_gt1 = 'pd_' + dataset.target_panel[1], dataset.target_panel[1]


fig, ax = plt.subplots(1, figsize=(4, 4), dpi=200)
sc.pl.embedding(rna_adata, basis='spatial', ax=ax, show=False, s=5)

mask = rna_adata.obs[key_pd0] == 1
ax.scatter(rna_adata.obsm['spatial'][mask, 0], rna_adata.obsm['spatial'][mask, 1], color='green',  s=0.1)
mask = rna_adata.obs[key_pd1] == 1
ax.scatter(rna_adata.obsm['spatial'][mask, 0], rna_adata.obsm['spatial'][mask, 1], color='black', s=1)

plt.title('NicheTrans')


In [ ]:
fig, ax = plt.subplots(1, figsize=(4, 4), dpi=200)
sc.pl.embedding(rna_adata, basis='spatial', ax=ax, show=False, s=5)

mask = rna_adata.obs[key_gt0] == 1
ax.scatter(rna_adata.obsm['spatial'][mask, 0], rna_adata.obsm['spatial'][mask, 1], color='green',  s=0.1)
mask = rna_adata.obs[key_gt1] == 1
ax.scatter(rna_adata.obsm['spatial'][mask, 0], rna_adata.obsm['spatial'][mask, 1], color='black', s=1)

plt.title('Ground Truth')